# **Hybrid Search & Retrieval**


## Agenda

| # | Section | What You'll Learn |
|---|---------|-------------------|
| 1 | The Search Problem | Why keyword search isn't enough |
| 2 | Dense Embeddings | Semantic similarity with neural nets |
| 3 | Sparse Embeddings & SPLADE | Learned lexical matching |
| 4 | Why Hybrid? | When dense fails, when sparse fails |
| 5 | Data Prep | Loading & exploring the ESCI dataset |
| 6 | Generating Embeddings | Creating both representations |
| 7 | Indexing with Qdrant | Storing vectors for fast retrieval |
| 8 | Hybrid Search | Running parallel queries | 10 min |
| 9 | Reciprocal Rank Fusion | Merging ranked lists intelligently |
| 10 | Evaluation & Takeaways | Did it actually work? |

---
## **Part 1: The Search Problem**

Imagine you're building a product search engine for an e-commerce site. A user types:

> **"quiet bathroom exhaust fan"**

### What goes wrong with traditional keyword (BM25/TF-IDF) search?

- ✅ It **will** find documents containing the exact words "quiet", "bathroom", "exhaust", "fan"
- ❌ It **won't** find a product titled *"WhisperCeiling 190 CFM Ventilation System — ultra silent operation"*
  - Same concept, completely different words!

### What goes wrong with purely semantic (dense vector) search?

- ✅ It **will** understand that "quiet" ≈ "silent" and "exhaust fan" ≈ "ventilation system"
- ❌ It **might miss** exact model numbers, brand names, or technical specs like "80 CFM"
  - If a user searches for *"revent 80 cfm"*, they want that **exact** product

### 💡 **The Solution**

> **Hybrid Search** = Dense (semantic understanding) + Sparse (precise lexical matching)  
> You get the best of both worlds.

Let's build this from scratch.

---
## **Part 2: Dense Embeddings**




A dense embedding converts text into a **fixed-size vector** where **every dimension has a non-zero value**.

```
"quiet bathroom fan"  →  [0.23, -0.15, 0.87, 0.02, ..., -0.41]  (1024 dimensions)
"silent exhaust vent"  →  [0.21, -0.14, 0.85, 0.03, ..., -0.39]  (1024 dimensions)
                            ↑ very similar vectors! (high cosine similarity)
```

### How does this work?

1. A transformer model (like BERT) reads the text
2. It outputs a **single vector** that captures the *meaning* of the entire text
3. Texts with similar meanings end up as **nearby points** in this high-dimensional space

### Key Properties

| Property | Value |
|----------|-------|
| Vector size | Fixed (e.g., 1024 for BGE-Large) |
| Values per dimension | All non-zero (dense) |
| What it captures | **Semantic meaning** — synonyms, paraphrases, concepts |
| Distance metric | Cosine similarity (angle between vectors) |
| Weakness | Can miss exact keyword matches |


---



### Model we'll use: `BAAI/bge-large-en-v1.5`
https://huggingface.co/BAAI/bge-large-en-v1.5
- 1024-dimensional embeddings
- Strong general-purpose English text embeddings
- Trained on massive text pairs with contrastive learning

**NOTE**: in the class we will run bge-small-en-v1.5 because it is faster to download


---
## **Part 3: Sparse Embeddings & SPLADE**



A **sparse embedding** also converts text into a vector, but:
- The vector size = **vocabulary size** (e.g., 30,522 for BERT tokenizer)
- **Most values are zero** — only tokens relevant to the text are "activated"

```
"quiet bathroom fan"  →  [0, 0, ..., 0.8(fan), 0, ..., 1.2(quiet), 0, ..., 0.9(bathroom), 0, ..., 0]
                          ↑ 30,522 dimensions, but only ~30-80 are non-zero
```

### Wait — isn't that just TF-IDF / Bag of Words?

**No!** And this is the crucial distinction. Classic sparse methods like TF-IDF only activate tokens that *literally appear* in the text.


---



### SPLADE (Sparse Lexical AnD Expansion)

SPLADE uses a **trained neural network** (masked language model) to:

1. ✅ Activate tokens that **appear** in the text (like TF-IDF)
2. ✅ **Also** activate *related tokens that DON'T appear* in the text — this is called **term expansion**

#### Example of Term Expansion

For the text *"Fastembed is a great library for text embeddings!"*, SPLADE produces:

```json
{
    "fast": 2.57,        ← appears in text ("Fastembed")
    "text": 1.87,        ← appears in text
    "library": 1.48,     ← appears in text
    "software": 0.71,    ← EXPANDED! (not in text, but related)
    "tool": 0.21,        ← EXPANDED!
    "database": 0.10,    ← EXPANDED!
    "wonderful": 0.01,   ← EXPANDED! (synonym of "great")
}
```

**The neural network *learned* that a "great library for embeddings" is related to "software", "tool", etc.!**

### Key Properties

| Property | Value |
|----------|-------|
| Vector size | Vocabulary size (30,522) |
| Non-zero values | ~30-80 per document (very sparse!) |
| What it captures | **Lexical matching + learned term expansion** |
| Distance metric | Dot product |
| Strength | Precise keyword matching + expansion |


---



Model we'll be using  `prithvida/Splade_PP_en_v1`
https://huggingface.co/prithivida/Splade_PP_en_v1

---
## **Part 4: Why to go Hybrid?**

| Query | Dense Search | Sparse (SPLADE) Search | Hybrid Search |
|-------|-------------|----------------------|---------------|
| *"revent 80 cfm"* (exact product) | ⚠️ Might return generic fans | ✅ Nails the exact model | ✅ |
| *"quiet bathroom ventilation"* (conceptual) | ✅ Understands the concept | ⚠️ Expansion helps, but may not bridge large vocabulary gaps | ✅ |
| *"silent exhaust for small room"* (mix) | ✅ Gets "silent" ≈ "quiet" | ✅ "exhaust" is a key term + expansion covers "small" | ✅✅ |

> **⚠️ Nuance on SPLADE:** Remember from Part 3 that SPLADE *does* expand terms — "ventilation" might activate "fan", "airflow", etc. So it won't completely miss conceptual queries the way raw TF-IDF would. The gap is narrower than with classic keyword search, but dense models still have the edge when the vocabulary shift is large (e.g., "quiet" → "whisper-rated", "bathroom" → "residential wet-area").

### The Hybrid Search Pipeline

```
Query: "revent 80 cfm"
         │
         ├──→ Dense Model (BGE)    ──→ Top-10 results by semantic similarity
         │                                    │
         ├──→ Sparse Model (SPLADE) ──→ Top-10 results by lexical match
         │                                    │
         └──→ Reciprocal Rank Fusion ◄────────┘
                      │
                      ▼
              Final Ranked Results
```

***Now let's build this, step by step.***

---
## **Part 5: Setup & Data Preparation**

### 5.1 Install Dependencies

In [1]:
!pip install -qU qdrant-client fastembed datasets transformers

You should consider upgrading via the '/Users/indu.admin/Desktop/InterviewKickstart/Classes/Week7_Nov8/code/venv/bin/python3 -m pip install --upgrade pip' command.


### 5.2 Imports

Let's import everything we need upfront. Don't worry if some of these are unfamiliar — we'll explain each one as we use it.

In [ ]:
import json

import numpy as np
import pandas as pd
from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    SparseVector,
    PointStruct,
    QueryRequest,
    SparseIndexParams,
    SparseVectorParams,
    VectorParams,
    ScoredPoint,
)
from transformers import AutoTokenizer

import fastembed
from fastembed import SparseEmbedding, SparseTextEmbedding, TextEmbedding

print(f"FastEmbed version: {fastembed.__version__}")

/Users/indu.admin/Desktop/InterviewKickstart/Classes/Week7_Nov8/code/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


FastEmbed version: 0.7.4


### 5.3 Load the Dataset

We'll use **Amazon ESCI** (Exact, Substitute, Complement, Irrelevant) — a real-world product search relevance dataset. Each query-product pair has a human-labeled relevance grade:

https://huggingface.co/datasets/tasksource/esci

| Label | Meaning | Example |
|-------|---------|--------|
| **Exact** | This is what the user wanted | Query: "80 cfm fan" → 80 CFM bathroom fan |
| **Substitute** | Acceptable alternative | Query: "80 cfm fan" → 110 CFM fan |
| **Complement** | Related, goes with it | Query: "80 cfm fan" → Duct tape |
| **Irrelevant** | Not what they wanted | Query: "80 cfm fan" → Kitchen blender |

This gives us **ground truth** to evaluate our search quality at the end!

In [3]:
#dataset = load_dataset("tasksource/esci", split="train")

In [4]:
from datasets import load_dataset

# 1. Load the dataset in streaming mode (no full download)
streaming_dataset = load_dataset("tasksource/esci", split="train", streaming=True)

# 2. Since we are streaming, we can't use len() directly without reading it,
# but we can take a fixed small amount (e.g., 1000 items as ~1% or any targeted limit)
subsample_size = 1000
streamed_subsample = streaming_dataset.take(subsample_size)

# 3. Convert only this small subset to a list or standard dataset
dataset = streamed_subsample
print(f"Successfully streamed and loaded {subsample_size} records without downloading the whole dataset!")

Successfully streamed and loaded 1000 records without downloading the whole dataset!


In [ ]:
from datasets import Dataset

# Convert the streamed 1000 items into a standard Dataset
# This brings only 1000 items into memory and enables dataset operations like .filter()
list_data = list(dataset.take(1000))
dataset = Dataset.from_list(list_data)

# Filter for US locale
dataset = dataset.filter(lambda x: x["product_locale"] == "us")
print(f"Subset dataset prepared successfully! Final size: {len(dataset)}")

In [ ]:
print(f"Total query-product pairs: {len(dataset)}")
print(f"\nFeatures: {dataset.column_names}")

### 5.4 Data Cleaning

Real data is messy! The same product can appear multiple times (paired with different queries). For our **product catalog** (the documents we search over), we need unique products.

In [ ]:
source_df = dataset.to_pandas() # convert a dataset object into a pandas DataFrame

In [ ]:
# Deduplicate products (same product can appear with different queries)
df = source_df.drop_duplicates(
    subset=["product_text", "product_title", "product_bullet_point", "product_brand"]
)

In [ ]:
# Drop rows with missing essential fields
df = df.dropna(subset=["product_text", "product_title", "product_bullet_point", "product_brand"])

In [ ]:
print(f"Unique products in catalog: {len(df)}")
print(f"Total search queries: {len(source_df)}")
print(f"\n→ On average, each product appears in {len(source_df) / len(df):.1f} query-product judgments")

In [ ]:
# Let's peek at what a product looks like
sample = df.iloc[0]
print(f"Product Title:  {sample['product_title']}")
print(f"Brand:          {sample['product_brand']}")
print(f"ESCI Label:     {sample['esci_label']}")
print(f"Query:          {sample['query']}")

In [ ]:
df['query'].unique()

In [ ]:
len(df['product_id'].unique())

In [ ]:
df

### 5.5 Create the Document Text

We concatenate relevant fields into a single text per product. This is what we'll embed.

> **💡 Tip:** What you include in the document text matters a lot! Title + description + bullet points gives the embedding model the richest signal.

In [ ]:
df["combined_text"] = (
    df["product_title"] + "\n" + df["product_text"] + "\n" + df["product_bullet_point"]
)

In [ ]:
df.shape

In [ ]:
# Preview
print(df["combined_text"].iloc[0][:800] + "...")
print(f"\nTotal documents to embed: {len(df)}")

---
## **Part 6: Generating Embeddings**

Now we create **two** vector representations for each product:
1. **Sparse** (SPLADE) — for precise lexical matching
2. **Dense** (BGE-Large) — for semantic understanding



### 6.1 Load the Models

In [ ]:
# Let's see all supported Dense Models
supported_models = (
    pd.DataFrame(TextEmbedding.list_supported_models())
    .sort_values("size_in_GB")
    .drop(columns=["sources", "model_file", "additional_files"])
    .reset_index(drop=True)
)
supported_models

In [ ]:
# Let's see all supported Sparse Models
(
    pd.DataFrame(SparseTextEmbedding.list_supported_models())
    .sort_values("size_in_GB")
    .drop(columns=["sources", "model_file", "additional_files"])
    .reset_index(drop=True)
)

Embedding Leaderboard: https://huggingface.co/spaces/mteb/leaderboard

In [ ]:
# Models we will use
sparse_model_name = "prithvida/Splade_PP_en_v1"
# dense_model_name = "BAAI/bge-large-en-v1.5" # larger but slower
dense_model_name = "BAAI/bge-small-en-v1.5" # smaller but faster

In [ ]:
# This triggers model download on first run (~500MB sparse + ~1.2GB dense)
sparse_model = SparseTextEmbedding(model_name=sparse_model_name, batch_size=32)
dense_model = TextEmbedding(model_name=dense_model_name, batch_size=32)

print("✅ Both models loaded!")

In [ ]:
# Helper functions
def make_sparse_embedding(texts: list[str]) -> list[SparseEmbedding]:
    return list(sparse_model.embed(texts, batch_size=32))

def make_dense_embedding(texts: list[str]):
    return list(dense_model.embed(texts))

### 6.2 Deep Dive: What Does a Sparse Embedding Look Like?

Let's embed a single sentence and **inspect the internals**. This is the most important cell in this section — it shows the "term expansion" magic of SPLADE.

In [ ]:
test_text = "InterviewKickstart is a great place to learn Applied AI!"

In [ ]:
sparse_embedding = make_sparse_embedding([test_text])

In [ ]:
print(f"Indices: {sparse_embedding[0].indices.shape}")
print(f"Values:  {sparse_embedding[0].values.shape}")

In [ ]:
sparse_embedding

In [ ]:
print(type(sparse_embedding[0].indices))
print(type(sparse_embedding[0].values))

In [ ]:
print(f"Non-zero entries: {len(sparse_embedding[0].indices)}")
print(f"Full vector size:  30,522")
print(f"Actual storage:    {sparse_embedding[0].indices.nbytes + sparse_embedding[0].values.nbytes} bytes")

In [ ]:
print(f"Input text: '{test_text}'")
print(f"\nSparse vector has {len(sparse_embedding[0].indices)} non-zero values")
print(f"Out of a vocabulary of 30,522 tokens")
print(f"→ That's {100 - len(sparse_embedding[0].indices)/30522*100:.2f}% sparse!")
print(f"   (only {len(sparse_embedding[0].indices)/30522*100:.2f}% of dimensions are active)")

The raw output is just indices and values — let's decode those indices back to actual words using the tokenizer:

In [ ]:
def get_tokens_and_weights(sparse_embedding, model_name) -> dict[str, float]:
    """Decode sparse embedding indices back to human-readable tokens."""
    # Find the HuggingFace tokenizer for this model
    tokenizer_source = None
    for model_info in SparseTextEmbedding.list_supported_models():
        if model_info["model"].lower() == model_name.lower():
            tokenizer_source = model_info["sources"]["hf"]
            break

    if tokenizer_source is None:
        raise ValueError(f"Model {model_name} not found in supported models.")

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
    token_weight_dict = {}
    for i in range(len(sparse_embedding.indices)):
        token = tokenizer.decode([sparse_embedding.indices[i]])
        weight = sparse_embedding.values[i]
        token_weight_dict[token] = weight

    # Sort by weight (highest first)
    return dict(sorted(token_weight_dict.items(), key=lambda item: item[1], reverse=True))

In [ ]:
tokens = get_tokens_and_weights(sparse_embedding[0], sparse_model_name)

print("🔍 SPLADE Token Weights for: '" + test_text + "'\n")
print(f"{'Token':<20} {'Weight':>8}  {'Source'}")
print("─" * 55)

# Words that actually appear in the text (approximately)
original_words = set(test_text.lower().replace("!", "").split())

for token, weight in tokens.items():
    # Check if token (or a form of it) appears in original text
    is_original = any(token.replace("##", "") in word for word in original_words)
    source = "← in original text" if is_original else "← EXPANDED (not in text!)"
    print(f"{token:<20} {weight:>8.4f}  {source}")

### ⭐ Key Observation

Look at the **EXPANDED** tokens above! SPLADE has learned that:
- "learn" is related to → `education`, `training`, `study`, `teaching`
- "great" expands to → `good`, `excellent`, `wonderful`, `nice`
- "AI" expands to → `artificial`, `intelligence`, `machine`, `technology`
- "interview" is related to → `job`, `career`, `hiring`

**This is what makes SPLADE fundamentally different from TF-IDF** — it uses a neural network to *learn* term associations, not just count word frequencies.

The `##` prefixed tokens are **WordPiece subwords** — BERT's way of handling words not in its core vocabulary. For example, `kick` + `##start` = `kickstart`.


### 6.3 Deep Dive: What Does a Dense Embedding Look Like?

In [ ]:
dense_embedding = make_dense_embedding([test_text])

In [ ]:
print(f"Dense vector shape: {dense_embedding[0].shape}")
print(f"Every dimension has a value (min: {dense_embedding[0].min():.4f}, max: {dense_embedding[0].max():.4f})")
print(f"\nFirst 10 values: {dense_embedding[0][:10].round(4)}")

### **Quick Comparison: Dense vs Sparse**

| Property | Dense (BGE-Large) | Sparse (SPLADE) |
|----------|------------------|----------------|
| **Vector size** | 1,024 (fixed) | 30,522 (vocab size) |
| **Non-zero values** | All 1,024 | ~30-80 |
| **Interpretable?** | ❌ No (abstract dimensions) | ✅ Yes (each dim = a word!) |
| **Captures synonyms?** | ✅ Inherently | ✅ Via term expansion |
| **Exact keyword match?** | ⚠️ Weak | ✅ Strong |

### 6.4 Embed the Full Catalog

Now let's generate both embedding types for all 176 products. FastEmbed uses **data parallelism** across CPU cores to speed this up.

In [ ]:
product_texts = df["combined_text"].tolist()
print(f"Embedding {len(product_texts)} products...")

In [ ]:
len(product_texts)

In [ ]:
%%time
print("⏳ Generating sparse embeddings (SPLADE)...")
df["sparse_embedding"] = make_sparse_embedding(product_texts)
print("✅ Done!")

In [ ]:
%%time
print("⏳ Generating dense embeddings (BGE-Large)...")
df["dense_embedding"] = make_dense_embedding(product_texts)
print("✅ Done!")

> **💡 Note on Wall Time vs CPU Time:** You'll likely see wall time is much less than CPU time.  
> This is because FastEmbed uses **multiple CPU cores in parallel** to process batches concurrently.  
> The speedup factor ≈ number of available CPU cores.

---
## **Part 7: Indexing with Qdrant**

We now have 176 products, each with two vector representations. We need a way to **quickly find the most similar vectors** to a query. But each vector type uses a **different search strategy**:

- **Dense vectors** → **Approximate Nearest Neighbor (ANN)** search using algorithms like HNSW (Hierarchical Navigable Small World). This trades a tiny bit of accuracy for massive speed gains in high-dimensional continuous spaces.
- **Sparse vectors** → **Inverted index** lookup, similar to how traditional search engines work. Each vocabulary token maps to a list of documents that activate it — Qdrant walks the query's non-zero tokens and aggregates matching documents.

**Qdrant** is a vector database that:
- Stores vectors + associated metadata ("payload")
- Builds the right index type for each vector automatically
- Supports **both dense and sparse** vectors in the same collection

We'll use in-memory mode for this demo. In production, you'd run Qdrant as a server (Docker or Cloud).

### 7.1 Create the Collection

In [ ]:
client = QdrantClient(":memory:")  # In-memory for demo; use url="http://localhost:6333" in production

collection_name = "esci"

client.create_collection(
    collection_name,
    # Dense vector config: 1024-dim with cosine similarity
    vectors_config={
        "text-dense": VectorParams(
            # size=1024,             # Must match model output dimension (large model)
            size=384, # small model
            distance=Distance.COSINE,  # Cosine similarity for dense vectors
        )
    },
    # Sparse vector config: no fixed size needed (it's dynamic)
    sparse_vectors_config={
        "text-sparse": SparseVectorParams(
            index=SparseIndexParams(
                on_disk=False,    # Keep in RAM for speed
            )
        )
    },
)

print(f"✅ Collection '{collection_name}' created with dense + sparse vector support")

### 7.2 Prepare & Upload Points

Each "point" in Qdrant consists of:
- **ID** — unique identifier
- **Vectors** — our dense + sparse embeddings (stored under named keys)
- **Payload** — metadata (product text, product ID, etc.)

In [ ]:
def make_points(df: pd.DataFrame) -> list[PointStruct]:
    """Convert DataFrame rows into Qdrant PointStruct objects."""
    sparse_vectors = df["sparse_embedding"].tolist()
    product_texts = df["combined_text"].tolist()
    dense_vectors = df["dense_embedding"].tolist()
    rows = df.to_dict(orient="records")

    points = []
    for idx, (text, sparse_vector, dense_vector) in enumerate(
        zip(product_texts, sparse_vectors, dense_vectors)
    ):
        # Convert sparse embedding to Qdrant's SparseVector format
        sparse_vector = SparseVector(
            indices=sparse_vector.indices.tolist(),
            values=sparse_vector.values.tolist()
        )

        point = PointStruct(
            id=idx,
            payload={
                "text": text,
                "product_id": rows[idx]["product_id"],
            },
            vector={
                "text-sparse": sparse_vector,   # Sparse vector under its named key
                "text-dense": dense_vector.tolist(),  # Dense vector under its named key
            },
        )
        points.append(point)
    return points


points = make_points(df)
print(f"Prepared {len(points)} points for indexing")

In [ ]:
# Upload to Qdrant
client.upsert(collection_name, points)
print(f"✅ {len(points)} points indexed in Qdrant!")

---
## **Part 8: Hybrid Search**

Here's where it all comes together. For a single query, we:
1. Generate **both** sparse and dense query vectors
2. Fire **two parallel searches** against Qdrant
3. Get **two separate ranked lists** of results

### 8.1 The Search Function

In [ ]:
def search(query_text: str, top_k: int = 10):
    """
    Perform hybrid search: run dense AND sparse search in parallel,
    return both result lists.
    """
    # Step 1: Embed the query with BOTH models
    query_sparse_vectors = make_sparse_embedding([query_text])
    query_dense_vector = make_dense_embedding([query_text])

    # Step 2: Fire two searches in a single batch request
    search_results = client.query_batch_points(
        collection_name=collection_name,
        requests=[
            # Search 1: Dense (semantic similarity)
            QueryRequest(
                query=query_dense_vector[0].tolist(),
                using="text-dense",
                limit=top_k,
                with_payload=True,
            ),
            # Search 2: Sparse (lexical matching)
            QueryRequest(
                query=SparseVector(
                    indices=query_sparse_vectors[0].indices.tolist(),
                    values=query_sparse_vectors[0].values.tolist(),
                ),
                using="text-sparse",
                limit=top_k,
                with_payload=True,
            ),
        ],
    )

    # Extract the point lists from QueryResponse objects
    return [search_results[0].points, search_results[1].points]

### 8.2 Run the Search

Let's search for **"revent 80 cfm"** — a specific product query where exact keyword matching matters.

In [ ]:
query_text = "revent 80 cfm"
search_results = search(query_text)

dense_results, sparse_results = search_results[0], search_results[1]

print(f"Query: '{query_text}'")
print(f"\n{'='*60}")
print(f"DENSE (Semantic) Results — Top 5:")
print(f"{'='*60}")
for i, point in enumerate(dense_results[:5]):
    title = point.payload['text'].split('\n')[0][:80]
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}")

print(f"\n{'='*60}")
print(f"SPARSE (Lexical/SPLADE) Results — Top 5:")
print(f"{'='*60}")
for i, point in enumerate(sparse_results[:5]):
    title = point.payload['text'].split('\n')[0][:80]
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}")

### 🤔 Discussion Point

Compare the two result lists above:
- Are they identical? Probably not!
- Which one caught exact "80 CFM" products better?
- Which one found semantically related products that use different words?

**This difference is exactly why we need a fusion strategy.**

---
## **Part 9: Reciprocal Rank Fusion (RRF)**

### **The Problem**

We have two ranked lists with **incompatible scores**:
- Dense search returns cosine similarity scores (-1 to 1, though typically positive for trained embeddings)
- Sparse search returns dot product scores (0 to ∞, since SPLADE weights are non-negative)

We **cannot** simply average these scores — they're on completely different scales!

### **The Solution:** Reciprocal Rank Fusion

RRF ignores the raw scores entirely and only uses **rank positions**. The formula:

$$\text{RRF\_score}(item) = \sum_{r \in \text{rank\_lists}} \frac{1}{\alpha + rank_r(item)}$$

Where:
- $\alpha$ = smoothing constant (typically 60)
- $rank_r(item)$ = position of the item in rank list $r$ (1-indexed)
- If the item is missing from a rank list, it gets `default_rank` = 1000

###  **Worked Example**

Suppose item "X" appears at **rank 1** in dense results and **rank 3** in sparse results:

```
RRF(X) = 1/(60+1) + 1/(60+3) = 0.0164 + 0.0159 = 0.0323
```

Item "Y" appears at **rank 2** in dense but is **missing** from sparse (gets rank 1000):

```
RRF(Y) = 1/(60+2) + 1/(60+1000) = 0.0161 + 0.0009 = 0.0170
```

→ **X ranks higher** because it appeared in both lists!

### **Why alpha = 60?**

The constant `alpha` controls how much the rank position matters:
- **Small alpha** → Top ranks get much higher scores (more aggressive)
- **Large alpha** → Scores are more uniform across ranks (more conservative)
- **60 is the standard** from the original RRF paper (Cormack et al., 2009) — it works well in practice

### 9.1 Let's First Verify with a Toy Example

In [ ]:
def rrf(rank_lists, alpha=60, default_rank=1000):
    """
    Reciprocal Rank Fusion (RRF).

    Takes multiple rank lists and produces a single fused ranking.
    Only considers RANK POSITION, not raw scores — making it safe to
    combine results from different scoring systems.

    Args:
        rank_lists: List of [(item_id, rank), ...] lists
        alpha: Smoothing constant (default=60, from Cormack et al. 2009)
        default_rank: Rank assigned to items not in a list (high = penalty)

    Returns:
        Sorted list of (item_id, rrf_score) tuples
    """
    all_items = set(item for rank_list in rank_lists for item, _ in rank_list)
    item_to_index = {item: idx for idx, item in enumerate(all_items)}

    # Matrix: rows = items, cols = rank lists, filled with default_rank
    rank_matrix = np.full((len(all_items), len(rank_lists)), default_rank)

    for list_idx, rank_list in enumerate(rank_lists):
        for item, rank in rank_list:
            rank_matrix[item_to_index[item], list_idx] = rank

    # RRF formula: sum of 1/(alpha + rank) across all lists
    rrf_scores = np.sum(1.0 / (alpha + rank_matrix), axis=1)

    sorted_indices = np.argsort(-rrf_scores)  # Descending
    sorted_items = [(list(item_to_index.keys())[idx], rrf_scores[idx]) for idx in sorted_indices]

    return sorted_items

In [ ]:
# Toy example to build intuition
rank_list1 = [("A", 1), ("B", 2), ("C", 3)]   # System 1 thinks A > B > C
rank_list2 = [("B", 1), ("C", 2), ("D", 3)]   # System 2 thinks B > C > D
rank_list3 = [("A", 2), ("D", 1), ("E", 3)]   # System 3 thinks D > A > E

fused = rrf([rank_list1, rank_list2, rank_list3])

print("Fused ranking (RRF):")
print(f"{'Item':<6} {'RRF Score':>10}  Explanation")
print("─" * 55)
for item, score in fused:
    # Count how many lists this item appears in
    appearances = sum(1 for rl in [rank_list1, rank_list2, rank_list3]
                      if any(i == item for i, _ in rl))
    print(f"{item:<6} {score:>10.6f}  (appears in {appearances}/3 lists)")

### ⭐ Key Takeaway

Notice that items appearing in **more lists** tend to rank higher. RRF naturally rewards **consensus** across rankers — if both dense and sparse search agree a result is good, it rises to the top.

### 9.2 Apply RRF to Our Search Results

In [ ]:
def rank_list(search_result: list[ScoredPoint]):
    """Convert Qdrant search results to (id, rank) pairs."""
    return [(point.id, rank + 1) for rank, point in enumerate(search_result)]


# Convert both result lists to rank lists
dense_rank_list = rank_list(search_results[0])
sparse_rank_list = rank_list(search_results[1])

print("Dense ranks:", dense_rank_list[:5], "...")
print("Sparse ranks:", sparse_rank_list[:5], "...")

In [ ]:
# Fuse the rankings!
rrf_rank_list = rrf([dense_rank_list, sparse_rank_list])

print(f"\n🏆 HYBRID SEARCH RESULTS for: '{query_text}'")
print(f"{'='*70}")

# Retrieve full records for the fused results
fused_records = client.retrieve(
    collection_name=collection_name,
    ids=[item[0] for item in rrf_rank_list]
)

# Create a lookup by ID
record_by_id = {r.id: r for r in fused_records}

for rank, (item_id, score) in enumerate(rrf_rank_list, 1):
    record = record_by_id[item_id]
    title = record.payload['text'].split('\n')[0][:75]

    # Check if this item was in dense, sparse, or both
    in_dense = any(id == item_id for id, _ in dense_rank_list)
    in_sparse = any(id == item_id for id, _ in sparse_rank_list)
    source = "BOTH" if (in_dense and in_sparse) else ("Dense only" if in_dense else "Sparse only")

    print(f"  {rank:>2}. [{source:<12}] {title}")

---
## **Part 10: Evaluation — Did It Work?**

Remember those ESCI labels? Let's check the ground truth relevance for our hybrid search results.

A perfect search engine would return **all Exact** labels for this query.

In [ ]:
ids = [item[0] for item in rrf_rank_list]

print(f"Query: '{query_text}'")
print(f"Results: {len(rrf_rank_list)} products")
print(f"\n{'Rank':<6} {'ESCI Label':<14} {'Product Title'}")
print("─" * 70)

label_counts = {}
for rank, idx in enumerate(ids, 1):
    label = df.iloc[idx]["esci_label"]
    title = df.iloc[idx]["product_title"][:60]
    emoji = {"Exact": "✅", "Substitute": "🔄", "Complement": "➕", "Irrelevant": "❌"}.get(label, "?")
    print(f"  {rank:<4} {emoji} {label:<12} {title}")
    label_counts[label] = label_counts.get(label, 0) + 1

print(f"\n{'='*70}")
print(f"📊 Summary:")
for label, count in sorted(label_counts.items()):
    print(f"   {label}: {count}/{len(ids)} ({count/len(ids)*100:.0f}%)")

### 🎉 Results Analysis

If you see all (or mostly) **Exact** labels — that's outstanding! With just out-of-the-box embeddings (no fine-tuning), hybrid search achieves high precision on this e-commerce query.

**Why this works so well for "revent 80 cfm":**
- **SPLADE** catches the exact keywords "80" and "cfm" with high weights
- **BGE-Dense** understands these are all bathroom ventilation products
- **RRF** boosts products that both systems agree on

---
## **Part 11: Agentic RAG Pipeline (with LangChain)**




Traditional RAG = retrieve → generate. **Agentic RAG** adds a reasoning loop:
```
User Query
    │
    ▼
🤖 Agent (LLM) — powered by LangChain's create_agent (ReAct loop)
    │
    ├──→ Reason: "What should I search for?"
    │
    ├──→ Act: hybrid_search("quiet bathroom fan 80 cfm")
    │         │
    │         ▼
    │    Observe: Retrieved Context
    │         │
    ├──→ Reason: "Is this enough? Should I search again with a different query?"
    │
    ├──→ Act: hybrid_search("ventilation fan with built-in light")  ← refinement
    │         │
    │         ▼
    │    Observe: More Context
    │
    └──→ Final Answer (grounded in retrieved products)
```

The agent **decides** what to search for, whether to refine, and when it has enough context — instead of blindly retrieving once.

### 11.1  Install Required Libraries

In [ ]:
!pip install -qU langchain langchain-openai langgraph

### 11.2 API key Setup

In [ ]:
import os
# os.environ["OPENAI_API_KEY"] = "sk-..."  # ← paste your key here

In [ ]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("api_key")

### 11.3 Define the hybrid search tool

In [ ]:
from langchain.tools import tool


@tool
def hybrid_search_tool(query: str, top_k: int = 5) -> str:
    """Search the product catalog using hybrid retrieval (dense semantic + sparse lexical),
    fused with Reciprocal Rank Fusion. This combines the best of keyword matching
    and semantic understanding.

    Use this tool whenever you need to find products. You can call it multiple times
    with different queries to refine results or explore different aspects.

    Args:
        query: The search query to run against the product catalog.
        top_k: Number of results to return (default 5).
    """
    results = search(query, top_k=top_k)
    dense_ranks = rank_list(results[0])
    sparse_ranks = rank_list(results[1])
    fused = rrf([dense_ranks, sparse_ranks])

    records = client.retrieve(
        collection_name=collection_name,
        ids=[item[0] for item in fused[:top_k]],
    )
    record_by_id = {r.id: r for r in records}

    output_lines = []
    for rank, (item_id, score) in enumerate(fused[:top_k], 1):
        rec = record_by_id[item_id]
        text = rec.payload["text"][:500]
        output_lines.append(f"[Result {rank} | RRF Score: {score:.6f}]\n{text}\n")

    return "\n---\n".join(output_lines) if output_lines else "No results found."


print(f"✅ Tool registered: {hybrid_search_tool.name}")

### 11.4 Create the agent

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """You are an intelligent product search assistant with access to an e-commerce product catalog indexed in Qdrant with hybrid (dense + sparse) embeddings.

## Your tool
You have one tool: `hybrid_search_tool` — it runs both semantic and lexical search, then fuses results with Reciprocal Rank Fusion.

## Your agentic workflow
1. **Analyze** the user's query — what are they really looking for?
2. **Search** with a well-crafted query using the hybrid search tool.
3. **Evaluate** — are the results relevant enough? If not, search again with a refined or decomposed query.
4. **Decompose** complex questions into multiple searches if needed.
5. **Synthesize** a final answer grounded in the retrieved products. Cite product names/titles.

## Rules
- ALWAYS search before answering — never make up products.
- You may call the tool MULTIPLE times with different queries to get better coverage.
- If no relevant products are found, say so honestly.
- Keep answers concise but informative.
"""

In [ ]:
agent = create_agent(
    model = "gpt-4o",
    tools=[hybrid_search_tool],
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Agentic RAG agent created!")

### 11.5 Test Agent

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "I need a quiet bathroom exhaust fan around 80 CFM. What are my options?"}]},
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            msg = data["messages"][-1]
            print(f"\n[{step}] {type(msg).__name__}")
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  🔧 {tc['name']}({tc['args']})")
            if msg.content:
                print(f"  {msg.content[:500]}")

In [ ]:
# Exact brand + model (sparse shines here)
result = agent.invoke({"messages": [{"role": "user", "content": "Do you have Broan-NuTone fans?"}]})
print(result["messages"][-1].content)

In [ ]:
# Conceptual / synonym-heavy (dense shines here)
result = agent.invoke({"messages": [{"role": "user", "content": "I want something to reduce moisture and mold in my shower area"}]})

print(result["messages"][-1].content)

In [ ]:
# Mixed — brand + concept (hybrid wins)
result = agent.invoke({"messages": [{"role": "user", "content": "Homewerks fan with a built-in LED light"}]})
print(result["messages"][-1].content)

In [ ]:
# Technical spec query (sparse catches numbers, dense catches intent)
result = agent.invoke({"messages": [{"role": "user", "content": "What's the most energy efficient exhaust fan under 1.0 sones?"}]})
print(result["messages"][-1].content)